# Větvící proces: vymře, nebo propukne?

*Nad rámec skripta — aplikace.*

---

Jeden nakažený nakazí náhodný počet dalších, každý z nich zase, a tak dál. Jeden zákazník doporučí službu náhodnému počtu známých. Jeden neutron vyvolá náhodný počet dalších štěpení. Všechno je to týž model: **větvící proces**.

Otázka je pokaždé stejná a má překvapivě ostrou odpověď:

> Vymře to, nebo poroste bez omezení?

Odpověď závisí na jediném čísle — na středním počtu potomků $R_0$ — a je **nespojitá**. Pro $R_0 \le 1$ proces vymře s jistotou, pro $R_0 > 1$ je naděje na přežití kladná. Žádný plynulý přechod, zlom v jediném bodě.

Tenhle sešit přináší do repozitáře dvě nové věci:

1. **Zpětnou vazbu**: dosud byly události nezávislé. Tady každá událost plodí další.
2. **Fázový přechod**: chování se v jednom bodě kvalitativně změní.

A má přesnou kontrolu: pravděpodobnost vymření je **nejmenší kořen rovnice $s = G(s)$**, kde $G$ je vytvořující funkce počtu potomků — tatáž $G$, kterou jsme použili na shluky škod v `aplikace/pojistovna_realisticky`.

In [ ]:
import math
import random

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


def poisson(lam):
    """Počet potomků. Pro malé λ součin rovnoměrných (cvičení 3), pro velké normální
    aproximace — ta je tu jen kvůli rychlosti, u velkých populací na přesnosti nezáleží."""
    if lam > 60:
        return max(0, int(round(random.gauss(lam, math.sqrt(lam)))))
    L, p, k = math.exp(-lam), 1.0, 0
    while True:
        p *= random.random()
        if p < L:
            return k
        k += 1

## 1. Model a přesné řešení

Začneme jedním nakaženým. Každý jedinec nezávisle nakazí $X$ dalších, kde $X$ má rozdělení s vytvořující funkcí

$$G(s) = E\,s^X = \sum_k P(X = k)\,s^k, \qquad R_0 = E X = G'(1).$$

Pro Poissonovo rozdělení je $G(s) = e^{R_0(s-1)}$.

### Proč je pravděpodobnost vymření kořenem $s = G(s)$

Označme $q$ pravděpodobnost, že celý proces vymře. První jedinec má $X$ potomků a **každá z jeho $X$ větví je nezávislá kopie téhož procesu**. Aby vymřel celek, musí vymřít všechny:

$$q = \sum_k P(X=k)\,q^k = G(q).$$

Takže $q$ je pevný bod $G$. Jedničkou je vždycky (vymřít „úplně" je formálně možné), ale platí věta: **hledaná pravděpodobnost je nejmenší nezáporný kořen**. A ten je menší než 1 právě tehdy, když $R_0 > 1$.

Kořen najdeme prostou iterací $s_{k+1} = G(s_k)$ od nuly — konverguje zdola k nejmenšímu kořenu.

In [ ]:
def G_poisson(s, R0):
    return math.exp(R0 * (s - 1))


def vymreni_presne(R0, iteraci=2000):
    """Nejmenší nezáporný kořen rovnice s = G(s), iterací od nuly.

    Pro R0 <= 1 je kořen roven 1 (vymření je jisté). Vracíme ho rovnou, protože
    přesně v kritickém bodě by iterace konvergovala tak pomalu, že by i po dvou
    tisících krocích vyšlo 0.999 — a to by v tabulce vypadalo jako nejistota,
    která tam není.
    """
    if R0 <= 1.0:
        return 1.0
    s = 0.0
    for _ in range(iteraci):
        s = G_poisson(s, R0)
    return s


def simuluj_vetveni(R0, maxgen=300, prah=2000):
    """Jeden běh. Vrací (vymřelo, počet generací, maximální velikost populace).

    Překročí-li populace práh, považujeme proces za přeživší — šance na vymření
    z tak velké populace je astronomicky malá (je to q^n pro n jedinců).
    """
    n, nejvetsi = 1, 1
    for g in range(maxgen):
        if n == 0:
            return True, g, nejvetsi
        if n > prah:
            return False, g, nejvetsi
        n = poisson(n * R0)        # součet n nezávislých Poissonů je Poisson(n·R0)
        nejvetsi = max(nejvetsi, n)
    return n == 0, maxgen, nejvetsi


M_ = 20_000
print(f"{'R0':>5s} {'simulace':>10s} {'přesně':>9s} {'rozdíl':>9s}")
R0_hodnoty = (0.8, 1.0, 1.2, 1.5, 2.0, 3.0)
sim_q, pres_q = [], []
for R0 in R0_hodnoty:
    vymrelo = sum(simuluj_vetveni(R0)[0] for _ in range(M_)) / M_
    presne = vymreni_presne(R0)
    sim_q.append(vymrelo); pres_q.append(presne)
    print(f"{R0:>5.1f} {vymrelo:>10.4f} {presne:>9.4f} {vymrelo - presne:>+9.4f}")

print("\nSedí všude kromě R0 = 1. Tam je vymření jisté (q = 1), ale trvá tak dlouho,")
print("že nám ho práh populace utne — viz oddíl 3.")

## 2. Fázový přechod

Zajímavé je, **jak** se pravděpodobnost vymření mění s $R_0$. Není to plynulý přechod — do $R_0 = 1$ je rovna jedné, pak začne klesat, a v tom bodě má zlom.

In [ ]:
R0_jemne = [0.5 + 0.05 * i for i in range(51)]
q_presne = [vymreni_presne(R0) for R0 in R0_jemne]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))

ax = axes[0]
ax.plot(R0_jemne, q_presne, "r-", lw=2, label="přesně (kořen $s = G(s)$)")
ax.plot(R0_hodnoty, sim_q, "o", ms=7, color="tab:blue", label="simulace")
ax.axvline(1.0, color="k", ls=":", lw=1)
ax.text(1.03, 0.45, "zlom v $R_0 = 1$", fontsize=9)
ax.set_xlabel("$R_0$ (střední počet potomků)")
ax.set_ylabel("pravděpodobnost vymření $q$")
ax.set_title("Fázový přechod: do 1 jistota, pak naděje", fontsize=10)
ax.legend(fontsize=8)

# grafické řešení s = G(s)
ax = axes[1]
ss = [i / 200 for i in range(201)]
ax.plot(ss, ss, "k--", lw=1, label="$s$")
for R0, barva in ((0.8, "tab:blue"), (1.0, "tab:orange"), (2.0, "tab:green")):
    ax.plot(ss, [G_poisson(s, R0) for s in ss], color=barva, label=f"$G(s)$, $R_0 = {R0}$")
    q = vymreni_presne(R0)
    ax.plot([q], [q], "o", color=barva, ms=7)
ax.set_xlabel("$s$")
ax.set_ylabel("$G(s)$")
ax.set_title("Kořen je průsečík $G(s)$ s přímkou $s$", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Pravý obrázek ukazuje, proč je zlom právě v jedničce. $G$ je konvexní a $G(1) = 1$, takže přímka $s$ ji protne pod jedničkou **právě tehdy**, když je $G$ v bodě 1 strmější než přímka, tedy $G'(1) = R_0 > 1$. Celá věta o vymření se vejde do jednoho obrázku.

## 3. Kritický případ $R_0 = 1$

Zvláštní je hranice. Při $R_0 = 1$ je vymření **jisté** — ale populace se v průměru nemění a dožití je neúměrně dlouhé. Střední doba do vymření je dokonce **nekonečná**, přestože vymření nastane s pravděpodobností 1.

To je přesně ten důvod, proč nám v tabulce výš vyšlo $0.99$ místo $1.00$: některé běhy jsme utli na prahu populace.

In [ ]:
print(f"{'R0':>6s} {'podíl vymřelých':>16s} {'přesně q':>10s} {'medián generací*':>18s} {'nejdelší běh*':>14s}")
for R0 in (0.8, 0.95, 1.0, 1.05, 1.2):
    delky = []
    vymrelo = 0
    for _ in range(4000):
        v, g, _ = simuluj_vetveni(R0, maxgen=500, prah=5000)
        vymrelo += v
        if v:
            delky.append(g)
    delky.sort()
    med = delky[len(delky) // 2] if delky else float("nan")
    print(f"{R0:>6.2f} {vymrelo / 4000:>16.4f} {vymreni_presne(R0):>10.4f} {med:>18.0f} {max(delky) if delky else 0:>14d}")

print("\n* jen z běhů, které vymřely")
print("\nU R0 = 1 je vymření jisté, ale některé větve se táhnou stovky generací.")
print("Střední doba do vymření je tam nekonečná — simulace to pozná tak, že")
print("maximum roste s počtem běhů a nikdy se neustálí.")

## 4. Co to znamená prakticky

Model je hrubý (nikdo se neuzdravuje, populace je nekonečná, všichni jsou stejní), ale dvě věci říká správně a obě jsou neintuitivní:

**Zásah těsně pod jedničku nestačí „skoro".** Snížit $R_0$ z 1.5 na 1.1 je obrovská práce a pravděpodobnost propuknutí klesne jen z 0.58 na 0.17. Sníží-li se ale pod 1, propuknutí přestane být možné úplně. Odměna za poslední kousek cesty je nesrovnatelně větší než za všechny předchozí.

**I při $R_0 > 1$ většina zavlečení vyhasne.** Při $R_0 = 1.5$ vymře zhruba 41 % řetězců samo od sebe. Takže z toho, že několik prvních zavlečení nikam nevedlo, **neplyne, že je nemoc neškodná** — plyne jen, že jsme zatím měli štěstí.

Podívejme se ještě na velikost epidemie u těch, které nevyhasnou.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))

ax = axes[0]
for R0, barva in ((0.9, "tab:blue"), (1.0, "tab:orange"), (1.3, "tab:red")):
    for _ in range(25):
        n, cesta = 1, [1]
        for _ in range(40):
            n = poisson(n * R0)
            cesta.append(n)
            if n == 0 or n > 3000:
                break
        ax.semilogy([max(v, 0.5) for v in cesta], lw=0.8, alpha=0.6, color=barva)
    ax.plot([], [], color=barva, label=f"$R_0 = {R0}$")
ax.set_xlabel("generace")
ax.set_ylabel("velikost populace (log)")
ax.set_ylim(0.4, 5000)
ax.set_title("Jednotlivé běhy: vyhasnout, nebo vybuchnout", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
R0 = 1.5
velikosti = []
for _ in range(8000):
    v, g, nejv = simuluj_vetveni(R0, maxgen=200, prah=3000)
    if v:
        velikosti.append(nejv)
ax.hist(velikosti, bins=range(1, 22), align="left", alpha=0.65, color="tab:green")
ax.set_xlabel("největší dosažená velikost populace")
ax.set_ylabel("počet běhů")
ax.set_title(f"Běhy, které při $R_0 = {R0}$ samy vyhasly ({len(velikosti)} z 8000)", fontsize=10)
plt.tight_layout()
plt.show()

print(f"Z běhů, které vyhasly, jich {sum(1 for v in velikosti if v <= 3) / len(velikosti):.0%} nikdy")
print(f"nepřesáhlo tři nakažené. Vyhasnutí se pozná brzy, nebo vůbec.")

## Shrnutí

| Pojem | Co si pamatovat |
|-------|-----------------|
| Větvící proces | každý jedinec nezávisle plodí $X$ potomků se stejným rozdělením |
| $R_0 = E X$ | jediné číslo, které rozhoduje o osudu procesu |
| Pravděpodobnost vymření | **nejmenší nezáporný kořen $s = G(s)$** |
| Fázový přechod | $R_0 \le 1$ ⟹ vymření jisté; $R_0 > 1$ ⟹ kladná naděje na přežití |
| Grafický důkaz | $G$ je konvexní, $G(1)=1$ ⟹ kořen pod 1 existuje právě pro $G'(1) > 1$ |
| Kritický případ | při $R_0 = 1$ je vymření jisté, ale střední doba do něj **nekonečná** |
| Trik se součtem | součet $n$ Poissonů je Poisson($n\lambda$) — celá generace jedním losováním |
| Praktický závěr | i při $R_0 > 1$ většina zavlečení vyhasne; z toho neplyne bezpečí |

Zpětná vazba je tu to nové: dosud byly události nezávislé, tady každá plodí další. Důsledkem je ostrý zlom — a ten je typický pro všechny systémy se zesilující zpětnou vazbou, od epidemií po řetězovou reakci.

### Zkuste si

1. Nahraďte Poissonovo rozdělení potomků geometrickým se stejným $R_0$. Vymření má pak **uzavřený vzorec** — odvoďte ho z $G(s) = \frac{p}{1-(1-p)s}$ a porovnejte se simulací.
2. Jak se změní $q$, když místo jednoho zavlečeného začnete s deseti? (Nápověda: větve jsou nezávislé.)
3. Spočítejte rozdělení **celkového** počtu nakažených (ne jen maxima) pro $R_0 < 1$. Střední hodnota má jednoduchý tvar $1/(1-R_0)$ — ověřte.
4. Přidejte očkování: podíl $v$ populace je imunní, takže efektivní $R_0' = R_0(1-v)$. Jaké $v$ je potřeba, aby $R_0' < 1$ pro $R_0 = 3$?
5. U $R_0 = 1$ změřte, jak roste maximální délka běhu s počtem opakování. Ustálí se?
6. Omezte populaci (nakazit lze jen z $N$ lidí) a sledujte, jak se neomezený růst změní na epidemickou křivku se špičkou.